# Extra??o das Transa??es do Bitcoin e An?lise com PySpark

In [ ]:
# https://pypi.org/project/findspark/
!pip install -q findspark
!pip install yfinance

In [ ]:
# Importa o findspark e inicializa
import findspark
findspark.init()

In [ ]:
# Imports
import pyspark                        
import pandas as pd                                            # manipula??o de dados
import numpy as np                                             # manipula??o de dados
import seaborn as sns                                          # gr?ficos
from matplotlib import pyplot as plt                           # gr?ficos
from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession, SQLContext
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.ml.linalg import Vectors
from pyspark.ml.feature import StringIndexer
from pyspark.ml.regression import LinearRegression
from pyspark.mllib.evaluation import RegressionMetrics
from pyspark.ml.linalg import Vectors
from pyspark.ml.feature import StringIndexer
from pyspark.ml.stat import Correlation
from pyspark.ml.feature import MinMaxScaler
from pyspark.ml.feature import VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator, CrossValidatorModel
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.evaluation import RegressionEvaluator

# Para extrair os dados da criptomoeda
import yfinance as yf
from datetime import datetime

In [ ]:
# Definir o s?mbolo do Bitcoin no Yahoo Finance (BTC-USD)
symbol = "BTC-USD"

# Definir o intervalo de datas (exemplo: 2017-01-01 at? 2023-12-31)
start_date = "2014-09-01"
end_date = datetime.now().strftime("%Y-%m-%d")  

# Obter os dados hist?ricos
data = yf.download(symbol, start = start_date, end = end_date)

# Exibir as primeiras linhas do DataFrame
print(data.head())

# Salvar os dados em um arquivo CSV
data.to_csv("dados/bitcoin_historico_yfinance.csv")


In [ ]:
# Formata??o das sa?das
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 400)
from matplotlib.axes._axes import _log as matplotlib_axes_logger
matplotlib_axes_logger.setLevel('ERROR')

In [ ]:
# Vers?es dos pacotes usados neste jupyter notebook
%reload_ext watermark
%watermark -a "Pedro Valadares J?nior" --iversions

In [ ]:
# Definindo semente aleat?ria (seed) para reprodutibilidade do notebook
rnd_seed = 23
np.random.seed = rnd_seed
np.random.set_state = rnd_seed

In [ ]:
# Criando o Spark Context(em Spark, tudo que crio ? um App)

sc = SparkContext(appName = 'Estudo_PySpark')


# O c?digo abaixo abre uma conex?o para o meu cluster. Observe que n?o identificamos o 
# cluster, quando n?o identificamos, ele cria automaticamente no cluster local(minha m?quina).

In [ ]:
spark_session = SparkSession.Builder().getOrCreate()

In [ ]:
spark_session

### LOADING

In [ ]:
df_spark = spark_session.read.csv('dados/bitcoin_historico_yfinance.csv', header = 'true', inferSchema = 'true')

In [ ]:
# Exibe as primeiras linhas
df_spark.show()

In [ ]:
# Retorna o tipo de cada coluna
df_spark.printSchema()

In [ ]:
# Exibe um sum?rio estat?stico
df_spark.describe().show()

In [ ]:
# Contando as linhas
linhas = df_spark.count()
print(f'Total de linhas: {linhas}')

In [ ]:
# Retornando os nomes das colunas do dataset
df_spark.columns

### DATA CLEANING

In [ ]:
# Excluindo valores nulos
df_spark = df_spark.na.drop()

In [ ]:
# Checando se os valore nulos foram excluidos
null_counts = df_spark.select(
    
    [sum(col(c).isNull().cast("int")).alias(c) for c in df_spark.columns]
)

null_counts.show()

# c representa o nome de cada coluna de df_spark
# col() ? uma fun??o pyspark que cria refer?ncia para a coluna com o nome c
# col(c).isNull() verifica se a coluna cont?m nulos ou n?o. (booleano)
# cast("int") converte o boleano em um n?mero inteiro (0 ou 1)
# sum() soma todos os valores para cada coluna, que agora s?o 1 ou 0.
# alias(c) ao resultado da soma usa o nome da coluna atual(original)

In [ ]:
df_spark.printSchema()

In [ ]:
# Renomeando o nome de uma coluna
df_spark = df_spark.withColumnRenamed('Adj Close', 'Adj_Close').withColumnRenamed('Price', 'Date')
df_spark.columns

### DATA TRANSFORMATION

In [ ]:
# Usando select() para retornar colunas espec?ficas
df_spark.select('Date', 'Open', 'Low', 'High').show()

In [ ]:
# Criando uma nova coluna com a diferen?a entre a alta e a baixa
df_spark.withColumn('High-Low', col("High") - col("Low")).show()

### Verificando as Datas de Transa??es

In [ ]:
# Registra o DataFrame como uma tabela tempor?ria
df_spark.createOrReplaceTempView("tabela_datas")

In [ ]:
data_amplitude = spark_session.sql(
    """
        SELECT 
            MIN(DATE(Date)) AS Primeira_Negociacao,
            MAX(DATE(Date)) AS Ultima_Negociacao
        FROM
            tabela_datas
    """)

data_amplitude.show()

In [ ]:
# Filtrando os dados de volumes de transa??es acima de de 20 milh?es.
df_spark.filter(df_spark['Volume'] > 20000000).show()

In [ ]:
# Finalize a sess?o spark antes de come?ar outra.
df_spark.stop